In [46]:
import pandas as pd
import zipfile

zip_path = "LLCP2024XPT.zip" 

with zipfile.ZipFile(zip_path) as z:
    #check file names inside zip
    print(z.namelist())
    
    with z.open(z.namelist()[0]) as f:
        df = pd.read_sas(f, format='xport')

['LLCP2024.XPT ']


In [47]:
df.head()

,_STATE,FMONTH,IDATE,IMONTH,IDAY,IYEAR,DISPCODE,SEQNO,_PSU,CTELENM1,...,_LCSCTSN,_LCSPSTF,DRNKANY6,DROCDY4_,_RFBING6,_DRNKWK3,_RFDRHV9,_FLSHOT7,_PNEUMO3,_AIDTST4
0,1.0,2.0,b'02282024',b'02',b'28',b'2024',1100.0,b'2024000001',2.024000e+09,1.0,...,NaN,9.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,1.0,2.0,2.0
1,1.0,2.0,b'02212024',b'02',b'21',b'2024',1100.0,b'2024000002',2.024000e+09,1.0,...,4.0,9.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,1.0,1.0,2.0
2,1.0,2.0,b'02212024',b'02',b'21',b'2024',1100.0,b'2024000003',2.024000e+09,1.0,...,4.0,2.0,1.0,1.000000e+02,2.0,1.400000e+03,1.0,NaN,NaN,2.0
3,1.0,2.0,b'02282024',b'02',b'28',b'2024',1100.0,b'2024000004',2.024000e+09,1.0,...,NaN,9.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,1.0,1.0,2.0
4,1.0,2.0,b'02212024',b'02',b'21',b'2024',1100.0,b'2024000005',2.024000e+09,1.0,...,3.0,9.0,2.0,5.397605e-79,1.0,5.397605e-79,1.0,NaN,NaN,2.0


In [48]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 457670 entries, 0 to 457669
Columns: 301 entries, _STATE to _AIDTST4
dtypes: float64(296), object(5)
memory usage: 1.0+ GB


In [50]:
df.columns

Index(['_STATE', 'FMONTH', 'IDATE', 'IMONTH', 'IDAY', 'IYEAR', 'DISPCODE',
       'SEQNO', '_PSU', 'CTELENM1',
       ...
       '_LCSCTSN', '_LCSPSTF', 'DRNKANY6', 'DROCDY4_', '_RFBING6', '_DRNKWK3',
       '_RFDRHV9', '_FLSHOT7', '_PNEUMO3', '_AIDTST4'],
      dtype='object', length=301)

As a team we discussed and narrowed down this larger file to a cleaner 6 different predictors that we saw were most important based on past research and medical resources. 

Columns to add to the model:
 - EXERANY2
- _BMI5CAT
- _SMOKER3
- MENTHLTH
- SSBFRUT3
- ALCDAY4

Here is the ground truth column to determine if there is diabetes or not:
DIABETE4

In [51]:
cols = [
    "EXERANY2",
    "_BMI5CAT",
    "_SMOKER3",
    "MENTHLTH",
    "SSBFRUT3",
    "ALCDAY4",
    "DIABETE4"
]

final_df = df[cols]

In [52]:
final_df.head()

,EXERANY2,_BMI5CAT,_SMOKER3,MENTHLTH,SSBFRUT3,ALCDAY4,DIABETE4
0,1.0,2.0,4.0,88.0,NaN,888.0,3.0
1,1.0,3.0,3.0,88.0,NaN,888.0,3.0
2,1.0,2.0,1.0,88.0,NaN,230.0,3.0
3,1.0,3.0,4.0,88.0,NaN,888.0,3.0
4,2.0,2.0,4.0,88.0,NaN,888.0,3.0


In [53]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 457670 entries, 0 to 457669
Data columns (total 7 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   EXERANY2  457667 non-null  float64
 1   _BMI5CAT  414633 non-null  float64
 2   _SMOKER3  457670 non-null  float64
 3   MENTHLTH  457667 non-null  float64
 4   SSBFRUT3  115316 non-null  float64
 5   ALCDAY4   418449 non-null  float64
 6   DIABETE4  457666 non-null  float64
dtypes: float64(7)
memory usage: 24.4 MB


Now we will clean for each column and drop values that are N/A, blank, don't know, not sure, refused

In [54]:
final_df = final_df.dropna()

In [55]:
def drop_values(df, column, values_to_drop):
    return df[~df[column].isin(values_to_drop)]

In [56]:
final_df = drop_values(final_df, "EXERANY2", [7., 9.]) #dropping "Donï¿½t know/Not Sure" and "Refused" from the Exercise in Past 30 Days column
final_df = drop_values(final_df, "_SMOKER3", [9.]) #dropping "Donï¿½t know/Refused/Missing" from the Computed Smoking Status column
final_df = drop_values(final_df, "MENTHLTH", [77., 99.]) #dropping "Donï¿½t know/Not sure" and "Refused" from the Number of Days Mental Health Not Good column
final_df = drop_values(final_df, "SSBFRUT3", [777., 999.]) #dropping "Donï¿½t know/Not sure" and "Refused" from the Sugar-Sweetened Beverage column
final_df = drop_values(final_df, "ALCDAY4", [777., 999.]) #dropping "Donï¿½t know/Not sure" and "Refused" from the Days in past 30 had alcoholic beverage column
final_df = drop_values(final_df, "DIABETE4", [7., 9.]) #dropping "Donï¿½t know/Not sure" and "Refused" from the (Ever told) you had diabetes


In [57]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 103703 entries, 5092 to 432651
Data columns (total 7 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   EXERANY2  103703 non-null  float64
 1   _BMI5CAT  103703 non-null  float64
 2   _SMOKER3  103703 non-null  float64
 3   MENTHLTH  103703 non-null  float64
 4   SSBFRUT3  103703 non-null  float64
 5   ALCDAY4   103703 non-null  float64
 6   DIABETE4  103703 non-null  float64
dtypes: float64(7)
memory usage: 6.3 MB
